# Enhanced S3 to COG Converter with Automatic AWS Authentication

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Support for multiple AWS authentication methods**

Author: Kyle Lesinger (Enhanced version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
import rioxarray as rxr
import s3fs
import fsspec
from rasterio.warp import calculate_default_transform, reproject, Resampling
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import re

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked,
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links

[drcs_activations OLD Directory](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/)

[VEDA docs for file naming conventions](https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html)

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:
EVENT_NAME = '202507_Flood_TX'
#old name
#under drcs_activations
PRODUCT_NAME = 'sentinel'

PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name='nasa-disasters', verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name='nasa-disasters', verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, 'nasa-disasters', PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 13 .tif files in the S3 bucket.


['drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2C_20250717_cloudMask_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2C_colorInfrared_20250617_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2C_trueColor_20250617_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2C_trueColor_20250717_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2_NDVI_Change_20250617_20250717.tif',
 'drcs_activations/202507_Flood_TX/sentinel/NW_CentralTX_S2C_colorInfrared_20250620_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/NW_CentralTX_S2C_colorInfrared_20250710_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/NW_CentralTX_S2C_trueColor_20250620_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/NW_CentralTX_S2C_trueColor_20250710_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/ReevesCountyTX_S2B_colorInfrared_20250618_merged.tif',
 'drcs_activations/202507_Flood_TX/sentinel/ReevesCountyTX_S2B_c

## Load TIF Files from DRCS Data
### This may assist with diagnosing any issues that occur if no files are found in the code block above

This cell loads the pre-analyzed DRCS activation data from `drcs_activations_tif_files.json` which contains a complete inventory of all .tif files in the NASA Disasters S3 bucket.

The code will:
1. Load the JSON file containing the file inventory
2. Parse the `PATH_OLD` variable to find the corresponding directory
3. Extract all .tif filenames from that directory
4. Store them in `files_to_process` for later use

In [6]:
# # Load the pre-analyzed DRCS TIF files data using imported functions
# # The JSON path is relative to the notebook location
# json_path = Path('../../s3-crawler/drcs_activations_tif_files.json')

# # Load DRCS data
# drcs_data = load_drcs_data(json_path)

# if drcs_data:
#     # Get TIF files from the specified PATH_OLD using the imported function
#     tif_files = get_tif_files_from_path(PATH_OLD, drcs_data, DIR_OLD_BASE)
    
#     if tif_files:
#         print(f"\n📁 Found {len(tif_files)} .tif files in {PATH_OLD}:")
#         print("\nFirst 10 files:")
#         for i, file in enumerate(tif_files[:10], 1):
#             print(f"  {i:2d}. {file}")
#         if len(tif_files) > 10:
#             print(f"  ... and {len(tif_files) - 10} more files")
        
#         # Get files with full paths using the imported function
#         files_to_process = get_files_with_full_paths(PATH_OLD, drcs_data, DIR_OLD_BASE, json_path)
#         print(f"\n✅ Files ready for processing. Stored in 'files_to_process' variable.")
#     else:
#         print(f"\n❌ No files found. Please check the PATH_OLD variable.")
#         files_to_process = []
# else:
#     print(f"\n❌ Could not load DRCS data.")
#     files_to_process = []

# files_to_process

In [7]:
# # Example: List available activation events using the imported function
# print("📂 Available activation events in DRCS data:")
# events = list_available_directories('drcs_activations', drcs_data, json_path)

# # Show first 10 events
# for event in events[:10]:
#     print(f"  - {event}")
# if len(events) > 10:
#     print(f"  ... and {len(events) - 10} more events")

# # Example: List subdirectories for a specific event
# print(f"\n📁 Subdirectories in {EVENT_NAME}:")
# subdirs = list_available_directories(f'drcs_activations/{EVENT_NAME}', drcs_data, json_path)
# for subdir in subdirs:
#     print(f"  - {subdir}")

# For these we can see three different types of files

1. WM = water mask
2. rgb = red green blue
3. WM_diff = water mask difference between dates

### We are going to need 2 different directories for these!!!

We will keep WaterMask (WM) and rgb as separate directories

In [7]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [6]:
# For simplicity, let's use python list comprehension to return the files
# We may need to rename them in different ways for different products
# We will do a similar process later

## NOTE --- We can actually use these objects since they have the same path as the s3 files. We will call them again later

cir = [f for f in keys if "color" in f]
print(cir)
cm = [f for f in keys if "cloud" in f]
print(cm)
true = [f for f in keys if "true" in f]
print(true)


['drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2C_colorInfrared_20250617_merged.tif', 'drcs_activations/202507_Flood_TX/sentinel/NW_CentralTX_S2C_colorInfrared_20250620_merged.tif', 'drcs_activations/202507_Flood_TX/sentinel/NW_CentralTX_S2C_colorInfrared_20250710_merged.tif', 'drcs_activations/202507_Flood_TX/sentinel/ReevesCountyTX_S2B_colorInfrared_20250618_merged.tif', 'drcs_activations/202507_Flood_TX/sentinel/ReevesCountyTX_S2B_colorInfrared_20250708_merged.tif']
['drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2C_20250717_cloudMask_merged.tif']
['drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2C_trueColor_20250617_merged.tif', 'drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2C_trueColor_20250717_merged.tif', 'drcs_activations/202507_Flood_TX/sentinel/NW_CentralTX_S2C_trueColor_20250620_merged.tif', 'drcs_activations/202507_Flood_TX/sentinel/NW_CentralTX_S2C_trueColor_20250710_merged.tif', 'drcs_activations/202507_Flood_TX/sentinel/ReevesCountyTX_S2B_tru

## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [9]:
def convert_date(date_str):
    """
    Convert to YYYY-MM-DD.
    
    Args:
        datetime_str: String like '20250731'
    
    Returns:
        String like '2025-07-31'
    """
    # Extract components
    year = date_str[0:4]
    month = date_str[4:6]
    day = date_str[6:8]
    
    # Format with dashes and colons, add Z for UTC
    return f"{year}-{month}-{day}"

# Test
date_str = '20250731'
result = convert_date(date_str)
print(result)

2025-07-31


In [11]:
def create_cog_filename(f, EVENT_NAME):
    """Create COG filename for NDVI files, handling single or dual dates."""
    # Extract directory, filename, and extension
    directory, filename = os.path.split(f)
    stem, ext = os.path.splitext(filename)

    # Find all 8-digit date patterns
    dates = re.findall(r"\d{8}", stem)

    # Remove dates from the stem
    stem_clean = re.sub(r"_?\d{8}", "", stem)

    if len(dates) == 1:
        # Single date
        cog_filename = f"{EVENT_NAME}_{stem_clean}_{convert_date(dates[0])}_day.tif"
    elif len(dates) == 2:
        # Two dates → comparison format
        cog_filename = f"{EVENT_NAME}_{stem_clean}_c{convert_date(dates[0])}_{convert_date(dates[1])}_day.tif"
    else:
        raise ValueError(f"Unexpected number of dates in filename: {filename}")

    return cog_filename


filter_str = 'NDVI'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202507_Flood_TX_CentralTX_S2_NDVI_Change_c2025-06-17_2025-07-17_day.tif


There's three types of NDVI files in this list, so I will handle renaming all three with conditional statements:

In [26]:
# f'{EVENT_NAME}_{"_".join(fsplit[0:2])}_{"_".join(fsplit[3:8])}_{convert_sentinel_datetime(fsplit[2])}.tif'

## Define COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with proper CRS and caching.

In [13]:
# # Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename, 
                                target_dir = "Sentinel-2/NDVI", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202507_Flood_TX_CentralTX_S2_NDVI_Change_c2025-06-17_2025-07-17_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202507_Flood_TX/sentinel
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-2/NDVI

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202507_Flood_TX

[1/1] Processing: drcs_activations/202507_Flood_TX/sentinel/CentralTX_S2_NDVI_Change_20250617_20250717.tif
   Output filename: 202507_Flood_TX_CentralTX_S2_NDVI_Change_c2025-06-17_2025-07-17_day.tif
   [MEMORY] Initial: 2418.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 4.00 MB
   [NODATA] Source nodata value: -3.4028234663852886e+38
   [CHUNKS] Processing 1204 chunks (43x28)
   [BAND 1/1] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-3.4028234663852886e+38, max=0.39909425377845764, center sample non-zero=999996/1000000
            Estimated data coverage: 80.3% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpz0d214u7_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpbjhxt5qa.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/NDVI/202507_Flood_TX_CentralTX_S2_NDVI_Change_c2025-06-17_2025-07-17_day.tif
   [MEMORY] Final: 3331.5 MB (Change: +912.9 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202507_Flood_TX_CentralTX_S2_NDVI_Change_c2025-06-17_2025-07-17_day.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-2/NDVI/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-2/NDVI/files_converted.csv
📁 COGs saved locally to: output/202507_Flood_TX

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-20T17:11:44.780058


## Check STATUS
[Disasters Bucket](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/)